# NGSO SLS - Slice A: Coverage Explorer

Interactive coverage/availability analysis for NGSO constellations (analytic Kepler+J2, H3 grid, single-owner sharding). Presets cover the **Reliance-Jio sizing scenarios**; **Custom (Walker)** gives full T/P/F + altitude + inclination for up to two shells.

**k (min sats in view)** = minimum number of satellites simultaneously above the min elevation for a cell to count as covered; **availability** = fraction of time that holds.

Run the cells top to bottom. Adjust parameters in **Controls**, then click **Run simulation** (or re-run the **Results** cell).

## Setup

In [ ]:
# === Setup: make ngso_sls importable (Colab + local Jupyter Lab) ===
# LOCAL JUPYTER (recommended): in a terminal, `pip install -e .` in the repo once, then this
#   cell is a no-op (it detects ngso_sls and skips clone/install entirely).
# COLAB: set REPO_URL to your remote. Private repo -> add a GitHub token in Colab 'Secrets'
#   named GITHUB_TOKEN (enable Notebook access).
# NOTE: after you push new code, do Runtime -> Restart runtime, then Run all — a running kernel
#   keeps already-imported modules, so code changes only take effect on a fresh kernel.
REPO_URL = "https://github.com/luca-aalyria/spacetime-sls.git"

import importlib, importlib.util, subprocess, sys, os, re


def _run(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.returncode != 0:
        print("$", cmd)
        print(p.stdout[-2000:])
        print(p.stderr[-3000:])
        raise RuntimeError(f"command failed (exit {p.returncode}) - see output above")


if importlib.util.find_spec("ngso_sls") is None:  # no-op on local Jupyter if already installed
    url = REPO_URL
    try:  # optional private-repo auth via Colab secret GITHUB_TOKEN
        from google.colab import userdata
        _tok = userdata.get("GITHUB_TOKEN")
        if _tok and url.startswith("https://github.com/"):
            url = url.replace("https://", f"https://{_tok}@")
    except Exception:
        pass
    repo_dir = re.sub(r"\.git$", "", os.path.basename(REPO_URL.rstrip("/"))) or "repo"
    if not os.path.isdir(repo_dir):
        _run(f"git clone {url} {repo_dir}")
    else:
        _run(f"git -C {repo_dir} pull --ff-only")  # update a stale clone on a fresh kernel
    _run(f"{sys.executable} -m pip install {os.path.abspath(repo_dir)}")
    sys.path.insert(0, os.path.abspath(repo_dir))
    importlib.invalidate_caches()

import ngso_sls
print("ngso_sls", ngso_sls.__version__)

## Imports

In [ ]:
# Imports
from dataclasses import replace
from datetime import datetime, timezone
import traceback
import matplotlib.pyplot as plt
import ipywidgets as w
from IPython.display import display, clear_output, HTML

from ngso_sls.presets import JIO_SCENARIOS
from ngso_sls.config import Shell, Constellation, TimeGrid, SimConfig
from ngso_sls.pipeline import run_coverage_h3
from ngso_sls.grids.aor import AORS
from ngso_sls.viz.plots import (
    plot_coverage_hexmap,
    plot_sats_in_view_hexmap,
    plot_availability_map,
    plot_sats_in_view_vs_latitude,
    plot_availability_hist,
)
from ngso_sls.io.csv_io import write_availability_csv


def _lbl(t):
    return w.HTML(f"<b>{t}</b>")

## Controls
Set parameters here. `Global` at high H3 resolution is heavy - start coarse.

In [ ]:
# Parameter controls (adjust here; run the Results cell to compute/redraw)
_s = {"description_width": "130px"}
_wide = {"description_width": "initial"}
_L = w.Layout(width="330px")

scenario = w.Dropdown(options=list(JIO_SCENARIOS) + ["Custom (Walker)"],
                      value="Full 1600 (dual shell)", description="Scenario", style=_s, layout=_L)
aor = w.Dropdown(options=list(AORS), value="India", description="Service area", style=_s, layout=_L)

# Custom Walker controls (used only when Scenario = "Custom (Walker)")
planes1 = w.IntSlider(value=40, min=1, max=60, description="S1 planes", style=_s, layout=_L)
spp1 = w.IntSlider(value=30, min=1, max=40, description="S1 sats/plane", style=_s, layout=_L)
phase1 = w.IntSlider(value=1, min=0, max=59, description="S1 phasing F", style=_s, layout=_L)
alt1 = w.FloatSlider(value=650, min=300, max=1500, step=10, description="S1 altitude km", style=_s, layout=_L)
inc1 = w.FloatSlider(value=48, min=0, max=90, step=1, description="S1 inclination", style=_s, layout=_L)
shell2_on = w.Checkbox(value=False, description="Add second shell")
planes2 = w.IntSlider(value=20, min=1, max=60, description="S2 planes", style=_s, layout=_L)
spp2 = w.IntSlider(value=20, min=1, max=40, description="S2 sats/plane", style=_s, layout=_L)
phase2 = w.IntSlider(value=7, min=0, max=59, description="S2 phasing F", style=_s, layout=_L)
alt2 = w.FloatSlider(value=650, min=300, max=1500, step=10, description="S2 altitude km", style=_s, layout=_L)
inc2 = w.FloatSlider(value=70, min=0, max=90, step=1, description="S2 inclination", style=_s, layout=_L)

# Analysis controls
min_elev = w.FloatSlider(value=25, min=5, max=45, step=1, description="Min elev deg", style=_s, layout=_L)
cell_res = w.IntSlider(value=3, min=1, max=5, description="H3 resolution", style=_s, layout=_L)
duration_min = w.FloatSlider(value=60, min=10, max=240, step=10, description="Duration min", style=_s, layout=_L)
step_s = w.FloatSlider(value=60, min=10, max=120, step=10, description="Time step s", style=_s, layout=_L)
k_cov = w.IntSlider(value=1, min=1, max=30, description="k (min sats in view)",
                    style=_wide, layout=w.Layout(width="360px"))
use_shard = w.Checkbox(value=True, description="Use sharding (faster, identical result)")
run_btn = w.Button(description="Run simulation", button_style="primary", icon="play")

controls = w.VBox([
    _lbl("Scenario & service area"),
    w.HBox([scenario, aor]),
    _lbl("Custom Walker (used only when Scenario = 'Custom (Walker)')"),
    w.HBox([planes1, spp1, phase1]),
    w.HBox([alt1, inc1]),
    shell2_on,
    w.HBox([planes2, spp2, phase2]),
    w.HBox([alt2, inc2]),
    _lbl("Analysis  (k = min # satellites in view for a cell to count as covered)"),
    w.HBox([min_elev, cell_res]),
    w.HBox([duration_min, step_s]),
    w.HBox([k_cov, use_shard]),
    run_btn,
])
display(controls)

## Results
Click **Run simulation** (or run this cell). Each output renders in its own box: coverage availability map, mean-satellites-in-view map, satellites-in-view vs latitude, availability histogram, an interactive Plotly map (best-effort), and `coverage_availability.csv`.

In [ ]:
# Results — click "Run simulation" above, or run this cell. Each output has its own box.
_box_layout = w.Layout(border="1px solid #ccc", padding="6px", margin="4px", min_height="60px")
out_log = w.Output(layout=_box_layout)
out_avail = w.Output(layout=_box_layout)
out_siv = w.Output(layout=_box_layout)
out_lat = w.Output(layout=_box_layout)
out_hist = w.Output(layout=_box_layout)
out_plotly = w.Output(layout=_box_layout)


def _titled(title, box):
    return w.VBox([_lbl(title), box])


def _build_constellation():
    if scenario.value == "Custom (Walker)":
        shells = [Shell("s1", planes1.value * spp1.value, planes1.value,
                        min(phase1.value, planes1.value - 1), alt1.value, inc1.value,
                        min_elev_user_deg=min_elev.value)]
        if shell2_on.value:
            shells.append(Shell("s2", planes2.value * spp2.value, planes2.value,
                                min(phase2.value, planes2.value - 1), alt2.value, inc2.value,
                                min_elev_user_deg=min_elev.value))
        return Constellation(tuple(shells))
    return Constellation(tuple(replace(s, min_elev_user_deg=min_elev.value)
                               for s in JIO_SCENARIOS[scenario.value].shells))


def _draw(box, make_fig):
    with box:
        clear_output(wait=True)
        try:
            make_fig()
            plt.show()
        except Exception:
            traceback.print_exc()


def _simulate(_=None):
    with out_log:
        clear_output(wait=True)
        try:
            cons = _build_constellation()
            total = sum(s.walker_T for s in cons.shells)
            shape = "; ".join(f"{s.walker_T}/{s.walker_P}/{s.walker_F} @{s.inclination_deg:g}°/{s.altitude_km:g}km"
                              for s in cons.shells)
            sim = SimConfig(cons,
                            TimeGrid(datetime(2026, 1, 1, tzinfo=timezone.utc),
                                     duration_s=duration_min.value * 60.0, step_s=step_s.value),
                            k_coverage=k_cov.value)
            print(f"Constellation: {scenario.value}  |  {total} sats  |  {shape}")
            print(f"Running over {aor.value} (H3 res {cell_res.value}, {duration_min.value:.0f} min "
                  f"@ {step_s.value:.0f}s, k={k_cov.value} min sats in view)...")
            res = run_coverage_h3(sim, AORS[aor.value], cell_res=cell_res.value,
                                  shard_res=(1 if use_shard.value else None), chunk_steps=10)
            a = res["availability"]
            print(f"cells={len(res['cells'])}  availability(k={k_cov.value}) mean={a.mean():.3f} "
                  f"min={a.min():.3f} max={a.max():.3f}  mean sats-in-view={res['sats_in_view_mean'].mean():.1f} "
                  f"(max {res['sats_in_view_mean'].max():.0f})")
            write_availability_csv(res, "coverage_availability.csv",
                                   {"scenario": scenario.value, "aor": aor.value, "total_sats": total,
                                    "k_coverage": k_cov.value, "seed": sim.seed,
                                    "step_s": sim.time_grid.step_s, "propagator": "KeplerJ2"})
            print("wrote coverage_availability.csv")
        except Exception:
            traceback.print_exc()
            return
    _draw(out_avail, lambda: plot_coverage_hexmap(res, title=f"Coverage availability (k={k_cov.value}) - {aor.value}"))
    _draw(out_siv, lambda: plot_sats_in_view_hexmap(res, title=f"Mean satellites in view - {aor.value}"))
    _draw(out_lat, lambda: plot_sats_in_view_vs_latitude(res))
    _draw(out_hist, lambda: plot_availability_hist(res))
    with out_plotly:
        clear_output(wait=True)
        try:
            geo = plot_availability_map(res, title=f"Coverage availability - {aor.value}")
            display(HTML(geo.to_html(include_plotlyjs="cdn", full_html=False, default_height="520px")))
        except Exception:
            traceback.print_exc()


run_btn.on_click(_simulate)
display(w.VBox([
    _titled("Run log", out_log),
    _titled("Coverage availability map", out_avail),
    _titled("Mean satellites-in-view map", out_siv),
    _titled("Satellites in view vs latitude", out_lat),
    _titled("Availability histogram", out_hist),
    _titled("Interactive map (Plotly - best effort)", out_plotly),
]))
_simulate()  # initial run so "Run all" shows output without clicking